# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [32]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [33]:
# Configuration values for the RAG pipeline
GEMMA_MODEL = "gemma-4-26b-a4b-it"
CHUNK_SIZE = 120
CHUNK_OVERLAP = 30
TOP_K = 3

ANSWER_RULES = """You answer questions about a document using ONLY the context passages provided.
- Each passage begins with its page number, for example [Page 2].
- After each fact you use, cite the page in brackets, for example (p. 2).
- If the answer is not in the context, reply exactly: "The answer could not be found in the document."
- Keep answers concise: 2 to 4 sentences."""

TEMPERATURE = 0.2
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print("Settings configured.")

Settings configured.


## Step 2: Connect to Gemma and load the embedding model

In [34]:
import os
from google import genai
from google.genai import types, errors
from sentence_transformers import SentenceTransformer
import numpy as np
import pypdf

# Yahan seedha apni API key daal di hai taaki koi error na aaye
API_KEY = "AQ.Ab8RN6IOaKGyOKW26tO-s9Pm8pN_8PPbtipqqjV11yfL2Lgo8w"
client = genai.Client(api_key=API_KEY)

# Load embedding model
embedder = SentenceTransformer(EMBED_MODEL_NAME)
knowledge_base = {"name": "None", "chunks": [], "vectors": []}
print("Connected to Gemma successfully & Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Connected to Gemma successfully & Embedding model loaded!


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [35]:
ANSWER_RULES = """You are an Open-Source Project Onboarding Assistant.
Answer developer questions using ONLY the provided documentation passages.
- Cite the file or section name for every instruction you provide.
- If the answer is not in the context, reply strictly: "The answer could not be found in the provided project documentation."
- Keep answers step-by-step and concise: 2 to 4 sentences."""

## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [39]:
import gradio as gr
import os

gr.close_all()

# Premium Theme Setup
theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="blue",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui", "sans-serif"]
)

APP_TITLE = "🌟 Nova: Open-Source Contributor Assistant"
APP_DESCRIPTION = "Your intelligent guide to onboarding, setting up, and contributing to open-source projects."

def on_upload(files):
    if not files:
        return "Status: No files uploaded."
    try:
        all_pages = []
        file_names = []

        for f in files:
            file_path = f if isinstance(f, str) else f.name
            base_name = os.path.basename(file_path)
            file_names.append(base_name)

            if file_path.endswith(".md") or file_path.endswith(".txt"):
                pages = read_markdown(file_path)
            else:
                pages = read_pdf(file_path)

            for p in pages:
                all_pages.append(f"[File: {base_name}] {p}")

        combined_names = ", ".join(file_names)
        build_knowledge_base(all_pages, combined_names)
        return f"✅ Ready! Loaded documents: **{combined_names}**"
    except Exception as e:
        raise gr.Error(f"Error reading file: {e}")

def on_ask(question):
    if not question.strip():
        raise gr.Error("Enter a question first.")
    reply, found = answer(question)
    sources = "\n\n".join(f"**Section {c['page']}** (Relevance Score: {c['score']:.2f})\n> {c['text']}" for c in found)
    return reply, sources

# Building the Premium Layout
with gr.Blocks(theme=theme, title="Nova Assistant") as demo:
    gr.Markdown(f"<h1 style='text-align: center; color: #4338ca;'>{APP_TITLE}</h1>")
    gr.Markdown(f"<p style='text-align: center; font-size: 16px; color: #6b7280;'>{APP_DESCRIPTION}</p>")
    gr.Markdown("---")

    with gr.Row():
        # LEFT COLUMN (Configuration)
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 📂 Project Documents")
            doc_input = gr.File(label="Upload Docs (.md, .txt, .pdf)", file_types=[".pdf", ".md", ".txt"], type="filepath", file_count="multiple")
            status_md = gr.Markdown("Status: **Awaiting documents...**")

            gr.Markdown("---")
            # Faltu headings hata di hain, sirf clean buttons rakhe hain
            checklist_btn = gr.Button("🚀 Generate Onboarding Checklist", variant="secondary")
            summarize_btn = gr.Button("📑 Summarize Setup & PR Rules", variant="secondary")

        # RIGHT COLUMN (Chat)
        with gr.Column(scale=2):
            gr.Markdown("### 💬 Ask the Assistant")
            question_box = gr.Textbox(label="Your Question", placeholder="e.g., How do I install the dependencies?", show_label=False, lines=2)
            ask_btn = gr.Button("Ask Question", variant="primary")

            # Example Queries ko Suggested Questions bana diya
            gr.Examples([
                "How do I install the dependencies?",
                "What is the process for submitting a pull request?",
                "Who won the cricket world cup?"
            ], inputs=question_box, label="💡 Suggested Questions")

            gr.Markdown("### 🤖 Answer")
            answer_md = gr.Markdown(elem_classes="box")

            with gr.Accordion("🔍 Retrieved Passages (Citations)", open=False):
                sources_md = gr.Markdown()

    doc_input.upload(on_upload, inputs=doc_input, outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_md])

    checklist_btn.click(lambda: "Give me a step-by-step onboarding checklist to start contributing to this project.", outputs=question_box).then(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    summarize_btn.click(lambda: "Summarize the setup instructions from README and the PR rules from CONTRIBUTING.", outputs=question_box).then(on_ask, inputs=question_box, outputs=[answer_md, sources_md])

demo.launch(share=True)

Closing server running on port: 7860


/tmp/ipykernel_1414/3559107494.py:50: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Nova Assistant") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c08d4ced5a4eb3d2d5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}  # populated by build_knowledge_base()


def read_markdown(path):
    """Returns a list of sections from a Markdown or text file."""
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    # Har heading (##) ko ek alag "page" ya chunk maante hain
    sections = content.split("\n## ")
    return [f"Section: {sec}" for sec in sections if sec.strip()]


def split_into_chunks(pages):
    """STAGE 1: split each page into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for page_number, text in enumerate(pages, start=1):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({"page": page_number, "text": " ".join(chunk_words)})
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def read_markdown(path):
    """Returns a list of sections from a Markdown or text file."""
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()
    # Heading (##) ke hisaab se split karo
    sections = content.split("\n## ")
    return [f"Section: {sec}" for sec in sections if sec.strip()]

def build_knowledge_base(pages, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(pages)
    if not chunks:
        raise ValueError("No text was found.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(pages)} pages, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)
    context = "\n\n".join(f"[Page {c['page']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is on **page {found[0]['page']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with the sample document.
print(build_knowledge_base(SAMPLE_PAGES, "Sample CS 101 syllabus"))
test_question = "What happens if I submit a lab late?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [page {c['page']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Sample CS 101 syllabus**: 4 pages, 4 chunks.

Question: What happens if I submit a lab late?
Answer: 1. Late submissions result in a 10% deduction per day for up to a maximum of 3 days, after which they receive a score of zero.
2. However, each student is granted two grace days per semester that can be used to submit any lab late without penalty.

Source: [Page 2] Grading.

Retrieved passages:
   [page 2, similarity 0.53] Grading. Weekly lab exercises: 20%. Two mini-projects: 30% (15% each). Midterm exam: 20%. ...
   [page 4, similarity 0.33] Policies. Attendance: students must attend at least 75% of lectures to be eligible for the...
   [page 3, similarity 0.28] Schedule. Weeks 1-2: variables, data types, and input/output. Weeks 3-4: conditions and lo...


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |